# 08.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/01-when-deep-learning-wins/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
images, labels = digits.images, digits.target          # 1,797 images, each 8×8
print(images.shape)
print(images[0].astype(int))

def hand_features(img):
    """Six features a person might design for digit images."""
    return [img.mean(),                                 # average ink
            img[:4].mean() - img[4:].mean(),            # top minus bottom
            np.abs(img - img[:, ::-1]).mean(),          # left-right asymmetry
            (img > 4).sum(),                            # number of inked pixels
            img[:, :4].mean() - img[:, 4:].mean(),      # left minus right
            img[2:6, 2:6].mean()]                       # ink in the centre

H = np.array([hand_features(i) for i in images])
P = images.reshape(len(images), -1)                     # 64 raw pixels per image

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def logreg_accuracy(features):
    a, b, c, d = train_test_split(features, labels, test_size=0.25, random_state=42, stratify=labels)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(a, c)
    return model.score(b, d)

(1797, 8, 8)
[[ 0  0  5 13  9  1  0  0]
 [ 0  0 13 15 10 15  5  0]
 [ 0  3 15  2  0 11  8  0]
 [ 0  4 12  0  0  8  8  0]
 [ 0  5  8  0  0  9  8  0]
 [ 0  4 11  0  1 12  7  0]
 [ 0  2 14  5 10 12  0  0]
 [ 0  0  6 13 10  0  0  0]]


## Exercise 1

In [3]:
def more_features(img):
    quarters = [img[:4, :4].mean(), img[:4, 4:].mean(), img[4:, :4].mean(), img[4:, 4:].mean()]
    rows = list(img.mean(axis=1))                       # ink in each of the 8 rows
    return hand_features(img) + quarters + rows

H2 = np.array([more_features(i) for i in images])
print(f"{H2.shape[1]} hand-made features → {logreg_accuracy(H2):.1%}")

18 hand-made features → 89.3%


Eighteen features reach about 89%, up from 68%, because they start to describe *where* the ink is. Keep going and you end up reinventing the pixels themselves. That is the point: the more information you keep, the better, and a model that learns from everything beats a person choosing summaries.

## Exercise 2

In [4]:
centre = images[:, 2:6, 2:6].reshape(len(images), -1)
print(f"16 centre pixels → {logreg_accuracy(centre):.1%}")
print(f"64 pixels        → {logreg_accuracy(P):.1%}")

16 centre pixels → 90.2%
64 pixels        → 97.8%


The centre alone scores about 90% against 97.8%: much of each digit's shape sits on the outer rings. Throwing input away rarely helps a model that can learn its own weights.

## Exercise 3

- **(a) Simpler model.** A customer table with meaningful columns; boosting or linear regression (Module 02, 05).
- **(b) Deep learning.** Free text; meaning is in word patterns. Module 09 uses Transformers and LLMs.
- **(c) Deep learning.** Photos; transfer learning on a few hundred labelled examples (08.9).
- **(d) Simpler model.** A short time series; seasonal baselines and Holt-Winters did best (Module 07).
- **(e) Deep learning.** Audio; speech recognition models are deep networks, usually used pretrained.